# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print(f"OpenAI key loaded: {'✓' if OPENAI_API_KEY else '✗'}")
print(f"Tavily key loaded: {'✓' if TAVILY_API_KEY else '✗'}")

OpenAI key loaded: ✓
Tavily key loaded: ✓


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
_chroma_client = chromadb.PersistentClient(path="chromadb")
_openai_client = OpenAI()

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
      - query: a question about game industry.

    You'll receive results as list. Each element contains:
      - Platform: like Game Boy, Playstation 5, Xbox 360...)
      - Name: Name of the Game
      - YearOfRelease: Year when that game was released for that platform
      - Description: Additional details about the game
    """
    response = _openai_client.embeddings.create(
        input=query,
        model="text-embedding-ada-002"
    )
    query_embedding = response.data[0].embedding

    collection = _chroma_client.get_collection("udaplay")
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=['documents', 'metadatas']
    )

    games = []
    if results['metadatas']:
        for metadata in results['metadatas'][0]:
            games.append({
                'Platform': metadata.get('Platform', ''),
                'Name': metadata.get('Name', ''),
                'YearOfRelease': metadata.get('YearOfRelease', ''),
                'Description': metadata.get('Description', '')
            })
    return games

#### Evaluate Retrieval Tool

In [5]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
      - question: original question from user
      - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
      - useful: whether the documents are useful to answer the question
      - description: description about the evaluation result
    """
    client = OpenAI()
    docs_text = "\n".join([str(doc) for doc in retrieved_docs])

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are an evaluation assistant. Evaluate if the documents are enough to respond to the query. Respond in JSON with exactly: 'useful' (boolean) and 'description' (string)."
            },
            {
                "role": "user",
                "content": f"Question: {question}\n\nDocuments:\n{docs_text}\n\nAre these documents sufficient?"
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

#### Game Web Search Tool

In [6]:
from tavily import TavilyClient

@tool
def game_web_search(question: str) -> str:
    """
    Semantic search: Finds most results in the vector DB
    args:
      - question: a question about game industry.
    """
    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    response = tavily_client.search(query=question, max_results=3)

    results = []
    for result in response.get('results', []):
        results.append({
            'title': result.get('title', ''),
            'content': result.get('content', ''),
            'url': result.get('url', '')
        })
    return str(results)

### Agent

In [7]:
udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions="""You are UdaPlay, an AI Research Agent specialized in the video game industry.

Your approach to answering questions:
1. Use retrieve_game to search the internal database for relevant game information
2. Use evaluate_retrieval to assess if the retrieved documents are sufficient
3. If evaluate_retrieval returns useful=False, use game_web_search to find information on the web
4. Provide a comprehensive, accurate answer based on the information gathered""",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0
)
print("UdaPlay agent created!")

UdaPlay agent created!


In [8]:
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print('='*60)
    run = udaplay_agent.invoke(question)
    final_state = run.get_final_state()
    messages = final_state.get("messages", [])
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content:
            print(f"Answer: {msg.content}")
            break


Question: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Answer: Pokémon Gold and Silver was released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon, introducing new regions, Pokémon, and gameplay mechanics.

Question: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __terminatio

### (Optional) Advanced

In [9]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes